# Data Processing


In [ ]:
# Standard library
import ast
import os
import random
import shutil
import time
import zipfile
from collections import defaultdict

# Numerical and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split, TensorDataset

# Torchvision
import torchvision
from torchvision import datasets, transforms, models

# Scikit-learn
import sklearn
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dataset 1 (original)
!unzip "/content/drive/MyDrive/Datasets/APS360_archive_1.zip"

In [ ]:
path = '/content/dataset'

# Transform data
transform_augment = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

# Load dataset
augment_dataset = torchvision.datasets.ImageFolder(root=path, transform=transform_augment)
dataset = torchvision.datasets.ImageFolder(root=path, transform=transform)

torch.manual_seed(42) # Ensure same seed for reproducible results

train_index, val_index = random_split(list(range(len(dataset))), [0.889, 0.111]) # 80, 10 split

train_set = Subset(augment_dataset, train_index)
val_set = Subset(dataset, val_index)

print(f'Training Set Size: {len(train_set)}')
print(f'Validation Set Size: {len(val_set)}')

img, label = train_set[0]
img = img.permute(1, 2, 0)
plt.title("First Training Sample")
plt.imshow(img)
plt.show()

In [ ]:
# Dataset 2 (new test set)
!unzip "/content/drive/MyDrive/Datasets/APS360_archive_2.zip"

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)              # Python RNG (affects random.shuffle/sample)
np.random.seed(SEED)           # If anything uses NumPy RNG
torch.manual_seed(SEED)        # PyTorch RNG
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

number_samples = len(val_set) // 4
extract_dir = '/content/new_test'
output_dir = '/content/full_test_set'

folder_map = {
    "11.Normal Fundus": "normal",
    "4.Moderate DR": "diabetic_retinopathy",
    "10.Glaucoma": "glaucoma",
    "7.Cataract": "cataract"
}

# Make directories fresh each run
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)
for new_name in folder_map.values():
    os.makedirs(os.path.join(output_dir, new_name), exist_ok=True)

# Deterministic sampling + copying
for old_name, new_name in folder_map.items():
    old_path = os.path.join(extract_dir, old_name)
    new_path = os.path.join(output_dir, new_name)

    # Sort to avoid filesystem-order randomness
    images = sorted(os.listdir(old_path))

    # Use a dedicated RNG with fixed seed for clarity
    rng = random.Random(SEED)
    picked = rng.sample(images, min(number_samples, len(images)))

    for image in picked:
        src_path = os.path.join(old_path, image)
        dest_path = os.path.join(new_path, image)
        shutil.copy2(src_path, dest_path)

print(f"Dataset created at {output_dir}")

transform = transforms.Compose([
    transforms.Resize(512),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

new_set = datasets.ImageFolder(root=output_dir, transform=transform)
print(f"New Testing Set Size: {len(new_set)}")

# The first image will now be the same every run
img, label = new_set[0]
img = img.permute(1, 2, 0)
plt.title("First Testing Sample")
plt.imshow(img)
plt.axis('off')
plt.show()

# Primary Model

In [ ]:
!pip install grad-cam

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, CyclicLR
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from sklearn.model_selection import KFold

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Model (exact 4-class head) + optional initial freezing
def create_model(num_classes=4, freeze_layers=True):
    model = models.efficientnet_b3(weights='EfficientNet_B3_Weights.DEFAULT')

    # Replace classifier to 4 classes
    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.classifier[1].in_features, num_classes)
    )

    if freeze_layers:
        for p in model.parameters():
            p.requires_grad = False
        for p in model.classifier.parameters():
            p.requires_grad = True
    else:
        for p in model.parameters():
            p.requires_grad = True

    return model.to(device)

In [ ]:
# Accuracy
@torch.no_grad()
def calculate_accuracy(model, data_loader):
    model.eval()
    correct, total = 0, 0
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total

In [ ]:
# Training (progressive unfreezing + LR modes)
def train_model(
    model,
    train_loader,
    val_loader,
    epochs=5,
    lr=1e-3,
    lr_scheduler_type="none",   # "none" | "cyclic" | "cosine"
    progressive_unfreeze=False  # unfreeze one feature block per epoch
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    # LR scheduler
    if lr_scheduler_type == "cyclic":
        scheduler = CyclicLR(optimizer, base_lr=1e-5, max_lr=lr, step_size_up=max(1, len(train_loader)//2), mode="triangular")
    elif lr_scheduler_type == "cosine":
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    else:
        scheduler = None

    total_blocks = len(model.features)  # EfficientNet-B3 has 9

    for epoch in range(epochs):
        # Progressive unfreezing: one block per epoch (from the end)
        if progressive_unfreeze:
            blocks_to_unfreeze = min(epoch + 1, total_blocks)
            for p in model.features[-blocks_to_unfreeze:].parameters():
                p.requires_grad = True
            # re-init optimizer to include newly trainable params
            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
            print(f"[Epoch {epoch+1}] Unfrozen feature blocks: {blocks_to_unfreeze}")
        else:
            blocks_to_unfreeze = "All (no freezing)"

        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            if scheduler and lr_scheduler_type == "cyclic":
                scheduler.step()
            running_loss += loss.item()

        if scheduler and lr_scheduler_type == "cosine":
            scheduler.step()

        val_acc = calculate_accuracy(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs}, Blocks: {blocks_to_unfreeze}, Loss: {running_loss:.4f}, Val Acc: {val_acc:.4f}")

    return model

In [ ]:
# Evaluation
def evaluate_model(model, test_loader):
    test_acc = calculate_accuracy(model, test_loader)
    print(f"Test Accuracy: {test_acc:.4f}")
    return test_acc

In [ ]:
# Grad-CAM Visualization for All Test Images (adaptive circular mask)
def generate_gradcam_all(model, data_loader, target_layer, class_names=["Normal", "Diabetic Retinopathy", "Cataract", "Glaucoma"]):
    model.eval()
    cam = GradCAM(model=model, target_layers=[target_layer])

    all_images, all_labels, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_images.append(images)
            all_labels.append(labels)
            all_preds.append(preds)

    all_images = torch.cat(all_images)
    all_labels = torch.cat(all_labels)
    all_preds = torch.cat(all_preds)

    for idx in range(len(all_images)):
        image = all_images[idx].unsqueeze(0)
        label = all_labels[idx].item()
        pred = all_preds[idx].item()

        targets = [ClassifierOutputTarget(label)]
        grayscale_cam = cam(input_tensor=image, targets=targets)[0]

        img_np = image.squeeze().cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)

        # Apply circular mask
        gray_img = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        gray_blurred = cv2.medianBlur(gray_img, 15)
        circles = cv2.HoughCircles(gray_blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=100,
                                   param1=50, param2=30, minRadius=50, maxRadius=0)
        mask = np.zeros(gray_img.shape, dtype=np.uint8)
        if circles is not None:
            circles = np.uint16(np.around(circles))
            for c in circles[0, :1]:
                cv2.circle(mask, (c[0], c[1]), c[2], 1, thickness=-1)
        else:
            h, w, _ = img_np.shape
            center = (w // 2, h // 2)
            radius = min(center) - 5
            cv2.circle(mask, center, radius, 1, thickness=-1)

        grayscale_cam = grayscale_cam * mask
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        # Show images
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img_np)
        axes[0].set_title(f"Original: {class_names[label]}")
        axes[0].axis("off")

        pred_status = "Correct" if pred == label else f"Wrong (Pred: {class_names[pred]})"
        axes[1].imshow(cam_image)
        axes[1].set_title(f"Grad-CAM: {pred_status}")
        axes[1].axis("off")
        plt.show()

In [ ]:
# Dataloaders (standard)
def create_dataloaders(train_set, val_set, test_set, batch_size=32):
    return (
        DataLoader(train_set, batch_size=batch_size, shuffle=True),
        DataLoader(val_set, batch_size=batch_size, shuffle=False),
        DataLoader(test_set, batch_size=batch_size, shuffle=False),
    )

In [ ]:
# K-Fold Integration
def _unwrap_base_dataset(dataset_like):
    """Return (base_dataset, base_indices) so we can re-index correctly for Subset or raw Dataset."""
    if isinstance(dataset_like, Subset):
        return dataset_like.dataset, list(dataset_like.indices)
    else:
        return dataset_like, list(range(len(dataset_like)))

def _subset_from_base(base_dataset, base_indices, pick_indices):
    """Make a Subset of base_dataset using the selected indices from the base index list."""
    # map k-fold local indices -> global indices into the base dataset
    mapped = [base_indices[i] for i in pick_indices]
    return Subset(base_dataset, mapped)

def k_fold_loop(
    train_or_full_dataset,
    n_splits=5,
    batch_size=32,
    epochs=5,
    lr=1e-3,
    freeze_layers=False,
    lr_scheduler_type="none",     # "none" | "cyclic" | "cosine"
    progressive_unfreeze=False
):
    base_dataset, base_indices = _unwrap_base_dataset(train_or_full_dataset)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold = 1
    val_accs = []
    val_losses = []
    last_model = None

    for train_idx, val_idx in kf.split(base_indices):
        print(f"\n========== Fold {fold}/{n_splits} ==========")
        ds_train = _subset_from_base(base_dataset, base_indices, train_idx)
        ds_val   = _subset_from_base(base_dataset, base_indices, val_idx)

        dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
        dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)

        model = create_model(freeze_layers=freeze_layers)

        # Train and get last epoch metrics
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

        if lr_scheduler_type == "cyclic":
            scheduler = CyclicLR(optimizer, base_lr=1e-5, max_lr=lr, step_size_up=5, mode="triangular")
        elif lr_scheduler_type == "cosine":
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
        else:
            scheduler = None

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            for inputs, labels in dl_train:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                if scheduler and lr_scheduler_type == "cyclic":
                    scheduler.step()
                running_loss += loss.item()

            if scheduler and lr_scheduler_type == "cosine":
                scheduler.step()

            val_acc = calculate_accuracy(model, dl_val)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss:.4f}, Val Accuracy: {val_acc:.4f}")

        # Save last epoch metrics for averaging later
        val_accs.append(val_acc)
        val_losses.append(running_loss)

        last_model = model
        fold += 1

    avg_acc = float(np.mean(val_accs)) if val_accs else 0.0
    avg_loss = float(np.mean(val_losses)) if val_losses else 0.0
    print(f"\n*** K-Fold Average Validation Accuracy: {avg_acc:.4f} ***")
    print(f"*** K-Fold Average Final Loss: {avg_loss:.4f} ***")

    return last_model, avg_acc, avg_loss

In [ ]:
# Training w/o K-Fold
# bs = 8, freeze/unfreeze = False, epochs = 5, lr = 0.001, scheduler = "cosine"
train_loader, val_loader, test_loader = create_dataloaders(
    train_set, val_set, new_set, batch_size=8
)

model = create_model(freeze_layers=False)
model = train_model(
    model, train_loader, val_loader,
    epochs=5, lr=1e-3,
    lr_scheduler_type="cosine",
    progressive_unfreeze=False
)

evaluate_model(model, test_loader)
generate_gradcam_all(model, test_loader, target_layer=model.features[-1])

In [ ]:
# Training w/ K-Fold
# Hyperparameters
n_splits = 5
batch_size = 8
epochs = 5
learning_rate = 1e-3

# Run K-Fold cross-validation
final_model, avg_val_acc, avg_val_loss = k_fold_loop(
    train_set,                # Only use training portion from Dataset 1
    n_splits=n_splits,
    batch_size=batch_size,
    epochs=epochs,
    lr=learning_rate,
    freeze_layers=False,
    lr_scheduler_type="cosine",
    progressive_unfreeze=False
)

test_loader = DataLoader(new_set, batch_size=batch_size, shuffle=False)
evaluate_model(final_model, test_loader)
generate_gradcam(final_model, test_loader, target_layer=final_model.features[-1])